# Профилирование AlexNet

Выполните ячейки сверху вниз на CUDA. Каждый профиль содержит пять измеряемых
шагов после четырёх начальных. Трейсы сохраняются в `artifacts/traces`.
Флаги в ячейке настроек включают дополнительные эксперименты.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd()
if not (ROOT / "utils" / "alexnet.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "utils" / "alexnet.py").is_file():
    raise RuntimeError("Откройте ноутбук из корня проекта или из папки notebooks.")
sys.path.insert(0, str(ROOT))

from utils.data_utils import load_training_data

DATA_DIR = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SEED = 52
BATCH_SIZE = 512

if not torch.cuda.is_available():
    raise RuntimeError("Для этих экспериментов нужна CUDA. Проверьте выбранное ядро.")
DEVICE = torch.device("cuda")
print(torch.cuda.get_device_name(DEVICE), "PyTorch", torch.__version__)

from utils.profiling import capture, benchmark_workers

WORKERS = 2
RUN_AUGMENT_ABLATION = False
RUN_WORKER_STUDY = False
data = load_training_data(DATA_DIR, SEED)

In [ ]:
capture(
    data, DEVICE, ARTIFACTS / "traces" / "training_trace_fp32.json",
    seed=SEED, batch_size=BATCH_SIZE, workers=WORKERS,
)

In [ ]:
capture(
    data, DEVICE, ARTIFACTS / "traces" / "training_trace_amp_fp16.json",
    seed=SEED, batch_size=BATCH_SIZE, workers=WORKERS, use_amp=True,
)

In [ ]:
if RUN_AUGMENT_ABLATION:
    for augment, label in ((True, "with"), (False, "without")):
        capture(
            data, DEVICE, ARTIFACTS / "traces" / f"training_trace_{label}_autoaugment.json",
            seed=SEED, batch_size=BATCH_SIZE, workers=0, augment=augment,
        )

In [ ]:
if RUN_WORKER_STUDY:
    import pandas as pd

    worker_results = []
    for workers in (0, 2, 4):
        ms = benchmark_workers(data, DEVICE, seed=SEED, batch_size=BATCH_SIZE, workers=workers)
        worker_results.append({"num_workers": workers, "ms_per_batch": ms})
        print(f"{workers} workers | {ms:.2f} мс/батч")

    pd.DataFrame(worker_results).to_csv(ARTIFACTS / "worker_benchmark.csv", index=False)